# B2 — BiLSTM + fastText cc.si.300 (v2)
T4 GPU. Embeddings ~1.5 GB (Drive-cached after the first run). Loading the .bin needs high RAM — if the session crashes at load, Runtime > Change runtime type > High-RAM.

In [ ]:
# cell 1 — environment (rerun if the session dies)
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!rm -rf repo
!git clone -b fix/v2-reproducibility-foundation https://github.com/ruwini01/Sinhala_English_Code_Mixed_Sentiment_Analysis.git repo
%cd /content/repo/ml
!pip install -q fasttext
!pip uninstall -y -q torchao

import os, shutil
SAVE = "/content/drive/MyDrive/Final_Reporing_Sentiment_Analysis/thesis_v2"
for sub in ("results", "checkpoints", "tokenized"):
    os.makedirs(f"{SAVE}/{sub}", exist_ok=True)

def bank(*paths, sub="results"):
    """Copy artifacts to Drive IMMEDIATELY — sessions die without warning."""
    for p in paths:
        if os.path.isdir(p):
            shutil.copytree(p, f"{SAVE}/{sub}/{os.path.basename(p)}", dirs_exist_ok=True)
        elif os.path.exists(p):
            shutil.copy2(p, f"{SAVE}/{sub}/")
        else:
            print("MISSING (not banked):", p)
    print("banked ->", f"{SAVE}/{sub}:", ", ".join(os.path.basename(p) for p in paths))

In [ ]:
# cell 2 — data: raw csv -> preprocess (splits are LOCKED in the repo)
import hashlib, os
RAW = "data/raw/singlish_mixed_sentiment_complete.csv"
os.makedirs("data/raw", exist_ok=True)

DRIVE_RAW = f"{SAVE}/singlish_mixed_sentiment_complete.csv"
if os.path.exists(DRIVE_RAW):
    shutil.copy2(DRIVE_RAW, RAW)
else:
    from google.colab import files
    files.upload()                      # pick the raw CSV from your PC
    os.replace("singlish_mixed_sentiment_complete.csv", RAW)
    shutil.copy2(RAW, DRIVE_RAW)        # bank the raw file itself

sha = hashlib.sha256(open(RAW, "rb").read()).hexdigest()
assert sha == "48ef313fb717411894a195a4bd488b52ce8431728f4c6c5d39643d1f4e097a7e", f"WRONG RAW FILE — sha256 {sha[:16]}... != v2.1 (see ml/DATA.md)"
print("raw sha256 verified: v2.1")

!python -m src.preprocess.quarantine
!python -m src.preprocess.clean_text
!python -m src.preprocess.language_id

In [ ]:
# cell 3 — fastText embeddings: Drive first, else download once and bank
import os
os.makedirs("data/embeddings", exist_ok=True)
EMB_GZ = f"{SAVE}/cc.si.300.bin.gz"
if os.path.exists(EMB_GZ):
    shutil.copy2(EMB_GZ, "data/embeddings/cc.si.300.bin.gz")
else:
    !wget -q -O data/embeddings/cc.si.300.bin.gz https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.si.300.bin.gz
    shutil.copy2("data/embeddings/cc.si.300.bin.gz", EMB_GZ)
!gunzip -kf data/embeddings/cc.si.300.bin.gz

# cell 3b — B2 run
!python -m src.train.run_bilstm_fasttext --fasttext data/embeddings/cc.si.300.bin
bank("results/bilstm_fasttext.json")

In [ ]:
# cell last — download to PC (put into ml/results/ locally, then commit)
from google.colab import files
files.download("results/bilstm_fasttext.json")